In [ ]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


: 

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os, certifi
from pymongo import MongoClient
mongo_client = MongoClient(
    os.getenv("ATLAS_URI"),
    tlsCAFile=certifi.where()
)
db = mongo_client["ecommerce_db"]
products_collection = db["products"]

In [ ]:
# Get all products 
all_products = list(products_collection.find())
print(f"Total products: {len(all_products)}")

Total products: 15003


In [ ]:
categories = products_collection.distinct("category")
for c in categories:
    print(c)
print(f"{len(categories)} categories found")

Bath & Shower
Detergents & Dishwash
Fragrance
Grocery & Gourmet Foods
Hair Care
Skin Care
6 categories found


In [ ]:
category_keywords = {}
for category in categories:
    unique_keywords = set()
    products = products_collection.find({"category": category})
    for product in products:
        if "keywords" in product and product["keywords"]:
            if isinstance(product["keywords"], list):
                unique_keywords.update(product["keywords"])
            else:
                unique_keywords.add(product["keywords"])
    category_keywords[category] = list(unique_keywords)

# Print keyword count by category
for category, keywords in category_keywords.items():
    print(f"{category}: {len(keywords)} unique keywords")

Bath & Shower: 4501 unique keywords
Detergents & Dishwash: 299 unique keywords
Fragrance: 2781 unique keywords
Grocery & Gourmet Foods: 9364 unique keywords
Hair Care: 3957 unique keywords
Skin Care: 21401 unique keywords


In [ ]:
# print(category_keywords)

In [ ]:
import pandas as pd
df2 = pd.read_csv("titles_to_categories.csv")
cat = set(list(df2["category_name"]))
cat

In [ ]:
# Create a balanced DataFrame instead of a new collection
balanced_data = []

for category in cat:
    # Sample approximately 100 documents from this category
    pipeline = [
        {"$match": {"category_name": category}},
        {"$sample": {"size": 100}}
    ]
    
    category_sample = list(df2[df2["category_name"] == category].sample(n=min(100, len(df2[df2["category_name"] == category]))).to_dict(orient="records"))
    print(f"Category '{category}': found {len(category_sample)} products")
    
    # Add to our data list
    balanced_data.extend(category_sample)

# Convert to DataFrame
balanced_df = pd.DataFrame(balanced_data)

# Save to CSV
balanced_df.to_csv("balanced_products.csv", index=False)
print(f"Saved {len(balanced_df)} products to balanced_products.csv")

# Check category distribution
category_counts = balanced_df['category_name'].value_counts()
print("\nCategory distribution in balanced DataFrame:")
print(category_counts)

Category 'Finger Toys': found 100 products
Category 'Garden Tools & Watering Equipment': found 100 products
Category 'Occupational Health & Safety Products': found 100 products
Category 'Bird & Wildlife Care': found 100 products
Category 'Handmade Jewellery': found 100 products
Category 'Decorative Artificial Flora': found 100 products
Category 'Kids' Art & Craft Supplies': found 100 products
Category 'Shaving  Hair Removal Products': found 100 products
Category 'Automotive Tires  Wheels': found 100 products
Category 'Office Products': found 100 products
Category 'Pens, Pencils & Writing Supplies': found 100 products
Category 'Perfumes & Fragrances': found 100 products
Category 'Computers & Tablets': found 100 products
Category 'Industrial & Scientific': found 100 products
Category 'Sony PSP Games, Consoles & Accessories': found 100 products
Category 'Darts & Dartboards': found 100 products
Category 'Mobile Phones & Communication': found 100 products
Category 'Golf Shoes': found 53 pro

In [ ]:
import pandas as pd

# Load the balanced_products.csv file
file_path = "balanced_products.csv"
df = pd.read_csv(file_path)

# Define the generalized mapping dictionary
specific_mapping = {
    "Bath & Shower": [
        "Bath & Body", 
        "Bath Products",
        "Bathroom Linen",
        "Shaving & Hair Removal Products",
        "Foot, Hand & Nail Care Products"
    ],
    "Detergents & Dishwash": [
        "Dishwashing Supplies",
        "Laundry Supplies",
        "Household Cleaning Supplies",
        "Household Cleaning Tools"
    ],
    "Fragrance": [
        "Perfumes & Fragrances",
        "Perfume & Cologne",
        "Home Fragrance",
        "Air Freshener Supplies"
    ],
    "Grocery & Gourmet Foods": [
        "Grocery",
        "Breakfast Cereal",
        "Beer, Wine & Spirits",
        "Coffee, Tea & Espresso",
        "International Food Market",
        "Luxury Food & Drink"
    ],
    "Hair Care": [
        "Hair Care",
        "Hair Care Products",
        "Shaving & Hair Removal Products"
    ],
    "Skin Care": [
        "Skin Care",
        "Skin Care Products",
        "Makeup",
        "Make-up",
        "Manicure & Pedicure Products",
        "Nail Polish & Nail Decoration Products"
    ]
}

# Function to map specific categories to generalized categories
def map_category(specific_category):
    for general_category, specific_list in specific_mapping.items():
        if specific_category in specific_list:
            return general_category
    return "Other"  # Default to "Other" if no match is found

# Apply the mapping to the category column
df['category_name'] = df['category_name'].apply(map_category)

# Save the updated DataFrame to a new CSV file
output_file_path = "balanced_products_generalized.csv"
df.to_csv(output_file_path, index=False)

print(f"Updated categories saved to {output_file_path}")

Updated categories saved to balanced_products_generalized.csv


In [ ]:
# Load the updated CSV file
df_generalized = pd.read_csv("balanced_products_generalized.csv")

# Calculate the size of each category
category_sizes = df_generalized['category_name'].value_counts()

# Print the sizes
print("Category sizes in the new CSV file:")
# Reduce the "Other" category to 1500 rows by randomly sampling
if "Other" in category_sizes.index and category_sizes["Other"] > 1500:
    other_rows = df_generalized[df_generalized["category_name"] == "Other"]
    sampled_other_rows = other_rows.sample(n=1500, random_state=42)
    df_generalized = pd.concat([df_generalized[df_generalized["category_name"] != "Other"], sampled_other_rows])

# Recalculate the size of each category
category_sizes = df_generalized['category_name'].value_counts()

# Save the updated DataFrame to a new CSV file
df_generalized.to_csv("products_updated.csv", index=False)
print(category_sizes)

Category sizes in the new CSV file:
category_name
Other                      1500
Grocery & Gourmet Foods     600
Bath & Shower               500
Skin Care                   500
Detergents & Dishwash       400
Fragrance                   300
Hair Care                   200
Name: count, dtype: int64


In [5]:
import pandas as pd

# Load the existing products_updated.csv file
updated_file_path = "products_updated.csv"
df_updated = pd.read_csv(updated_file_path)

# Load the products.csv file
products_file_path = "products.csv"
df_products = pd.read_csv(products_file_path)

# Get the current category counts
category_counts = df_updated['category_name'].value_counts()
print("Current category counts:")
print(category_counts)

# Create a list to store additional products
additional_products = []

# For each category, ensure it has exactly 1500 products
for category in category_counts.index:
    current_count = category_counts[category]
    
    if current_count < 1500:
        # Calculate how many more products are needed
        needed_count = 1500 - current_count
        print(f"Need {needed_count} more products for {category}")
        
        # Find matching products in products.csv (matching based on appropriate category mapping)
        # Get category name that matches in products.csv
        category_in_products = category
        
        # Filter products from products.csv for the specific category
        category_products = df_products[df_products['Category'] == category_in_products]
        
        if len(category_products) > 0:
            # Sample the required number of products (or all available if less than needed)
            sampled_products = category_products.sample(n=min(needed_count, len(category_products)), random_state=42)
            
            # Create a new dataframe with the correct column structure
            new_products = pd.DataFrame({
                'title': sampled_products['Product Description'],
                'category_name': sampled_products['Category']
            })
            
            # Add the sampled products to our list
            additional_products.append(new_products)
            
            print(f"Added {len(new_products)} products for {category}")
        else:
            print(f"No products found in products.csv for category {category}")
    
    elif current_count > 1500:
        # If we have more than 1500, reduce to exactly 1500
        print(f"Reducing {category} from {current_count} to 1500 products")
        df_updated = pd.concat([
            df_updated[df_updated['category_name'] != category],
            df_updated[df_updated['category_name'] == category].sample(n=1500, random_state=42)
        ])

# Combine all additional products
if additional_products:
    additional_df = pd.concat(additional_products, ignore_index=True)
    # Append the additional products to the updated DataFrame
    balanced_df = pd.concat([df_updated, additional_df], ignore_index=True)
else:
    balanced_df = df_updated.copy()

# Verify the final distribution
final_counts = balanced_df['category_name'].value_counts()
print("\nFinal category distribution:")
print(final_counts)

# Save the balanced DataFrame to a new CSV file
balanced_file_path = "products_balanced.csv"
balanced_df.to_csv(balanced_file_path, index=False)
print(f"\nBalanced dataset saved to {balanced_file_path}")

Current category counts:
category_name
Other                      1500
Grocery & Gourmet Foods     600
Bath & Shower               500
Skin Care                   500
Detergents & Dishwash       400
Fragrance                   300
Hair Care                   200
Name: count, dtype: int64
Need 900 more products for Grocery & Gourmet Foods
Added 900 products for Grocery & Gourmet Foods
Need 1000 more products for Bath & Shower
Added 1000 products for Bath & Shower
Need 1000 more products for Skin Care
Added 1000 products for Skin Care
Need 1100 more products for Detergents & Dishwash
Added 197 products for Detergents & Dishwash
Need 1200 more products for Fragrance
Added 1200 products for Fragrance
Need 1300 more products for Hair Care
Added 1300 products for Hair Care

Final category distribution:
category_name
Fragrance                  1500
Bath & Shower              1500
Grocery & Gourmet Foods    1500
Hair Care                  1500
Skin Care                  1500
Other             

In [1]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Check if dataset exists
file_path = "products_balanced.csv"
if not os.path.exists(file_path):
    print(f"Error: {file_path} file not found!")
    exit(1)

# 1. Load your dataset (description → category)
print("Loading and preparing dataset...")
df = pd.read_csv(file_path)  # Columns: ["title", "category_name"]

# Basic dataset info
print(f"Dataset loaded with {len(df)} products across {df['category_name'].nunique()} categories")
print(f"Top 5 categories by count:")
print(df['category_name'].value_counts().head())

# 2. Text preprocessing
def preprocess_text(text):
    """Basic text preprocessing"""
    if isinstance(text, str):
        return text.lower().strip()
    return ""

df['processed_title'] = df['title'].apply(preprocess_text)

# 3. Split data for evaluation (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category_name'])

# Reset indices after splitting to avoid KeyError
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training set: {len(train_df)} samples, Test set: {len(test_df)} samples")

# 4. Convert descriptions (titles) to embeddings
print("Generating embeddings with SentenceTransformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert DataFrame series to list first to avoid index issues
train_texts = train_df['processed_title'].tolist()
train_embeddings = model.encode(train_texts, show_progress_bar=True)

# 5. Train KNN model
print("Training KNN classifier...")
knn = KNeighborsClassifier(n_neighbors=3, metric='cosine')
knn.fit(train_embeddings, train_df['category_name'])

# 6. Define prediction function with improved confidence calculation
def predict_category(query, include_neighbors=False):
    """
    Predict category for a query text.
    
    Args:
        query (str): Text to classify
        include_neighbors (bool): Whether to return neighbor details
        
    Returns:
        Tuple of (category, confidence) or (category, confidence, neighbors_info)
    """
    try:
        # Preprocess the query
        processed_query = preprocess_text(query)
        
        # Generate embedding
        query_embedding = model.encode([processed_query])
        
        # Get predictions and distances
        distances, indices = knn.kneighbors(query_embedding)
        distances = distances[0]
        indices = indices[0]
        
        # Get predicted category
        category = knn.predict(query_embedding)[0]
        
        # Calculate confidence (cosine distance ranges from 0-2)
        max_distance = 2.0
        confidence = 1 - (np.mean(distances) / max_distance)
        
        # Optionally return neighbor details
        if include_neighbors:
            neighbors_info = []
            for idx, dist in zip(indices, distances):
                neighbors_info.append({
                    'title': train_df.iloc[idx]['title'],
                    'category': train_df.iloc[idx]['category_name'],
                    'distance': dist,
                    'similarity': 1 - (dist / max_distance)
                })
            return category, confidence, neighbors_info
        
        return category, confidence
        
    except Exception as e:
        print(f"Error during prediction: {e}")
        if include_neighbors:
            return "Unknown", 0.0, []
        return "Unknown", 0.0

# 7. Evaluate model on test set
print("Evaluating model on test set...")
test_texts = test_df['processed_title'].tolist()  # Convert to list to avoid index issues
test_embeddings = model.encode(test_texts, show_progress_bar=True)
predictions = knn.predict(test_embeddings)

accuracy = accuracy_score(test_df['category_name'], predictions)
print(f"Model accuracy: {accuracy:.4f}")

c:\Users\Nikhil\Downloads\rag\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and preparing dataset...
Dataset loaded with 9589 products across 7 categories
Top 5 categories by count:
category_name
Bath & Shower              1500
Grocery & Gourmet Foods    1500
Other                      1500
Fragrance                  1499
Hair Care                  1497
Name: count, dtype: int64
Training set: 7671 samples, Test set: 1918 samples
Generating embeddings with SentenceTransformer...


Batches: 100%|██████████| 240/240 [01:00<00:00,  3.96it/s]


Training KNN classifier...
Evaluating model on test set...


Batches: 100%|██████████| 60/60 [00:14<00:00,  4.18it/s]


Model accuracy: 0.7993


In [7]:
import joblib
import os
import json

# Create a directory to save the model files
model_dir = "knn_product_classifier"
os.makedirs(model_dir, exist_ok=True)

# 1. Save the KNN model
print("Saving KNN model...")
joblib.dump(knn, f"{model_dir}/knn_model.joblib")

# 2. Save the SentenceTransformer model
print("Saving SentenceTransformer model...")
model_save_path = f"{model_dir}/sentence_transformer"
model.save(model_save_path)

# 3. Save metadata about the model and categories
metadata = {
    "model_info": {
        "n_neighbors": knn.n_neighbors,
        "metric": knn.metric,
        "embedding_model": "all-MiniLM-L6-v2",
        "embedding_dimension": train_embeddings.shape[1],
    },
    "categories": train_df['category_name'].unique().tolist(),
    "category_counts": train_df['category_name'].value_counts().to_dict(),
    "training_samples": len(train_df),
    "accuracy": float(accuracy),
}

with open(f"{model_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# 4. Create a loader script to use the model
with open(f"{model_dir}/predict.py", "w") as f:
    f.write("""
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
import os

class ProductClassifier:
    def __init__(self, model_dir="."):
        # Load the KNN model
        self.knn = joblib.load(os.path.join(model_dir, "knn_model.joblib"))
        
        # Load the SentenceTransformer model
        self.model = SentenceTransformer(os.path.join(model_dir, "sentence_transformer"))
    
    def preprocess_text(self, text):
        "Basic text preprocessing"
        if isinstance(text, str):
            return text.lower().strip()
        return ""
    
    def predict(self, query, include_confidence=True):
        # Preprocess the query
        processed_query = self.preprocess_text(query)
        
        # Generate embedding
        query_embedding = self.model.encode([processed_query])
        
        # Get predictions and distances
        distances, indices = self.knn.kneighbors(query_embedding)
        distances = distances[0]
        
        # Get predicted category
        category = self.knn.predict(query_embedding)[0]
        
        if include_confidence:
            # Calculate confidence (cosine distance ranges from 0-2)
            max_distance = 2.0
            confidence = 1 - (np.mean(distances) / max_distance)
            return category, confidence
            
        return category

# Example usage
if __name__ == "__main__":
    classifier = ProductClassifier()
    
    # Test with some examples
    examples = [
        "wireless earbuds with noise cancellation",
        "anti-aging face cream with retinol",
        "smartphone case for iPhone 13",
        "kitchen knife set stainless steel"
    ]
    
    for query in examples:
        category, confidence = classifier.predict(query)
        print(f"Query: '{query}'")
        print(f"Predicted: {category} (Confidence: {confidence:.2f})")
        print()
    
    # Interactive mode
    print("Enter a product description to classify (or 'q' to quit):")
    while True:
        query = input("> ")
        if query.lower() == 'q':
            break
        
        category, confidence = classifier.predict(query)
        print(f"Predicted: {category} (Confidence: {confidence:.2f})")
""")

print(f"\nModel saved successfully to '{model_dir}/' directory!")
print(f"To use the saved model, run: python {model_dir}/predict.py")

Saving KNN model...
Saving SentenceTransformer model...

Model saved successfully to 'knn_product_classifier/' directory!
To use the saved model, run: python knn_product_classifier/predict.py


In [10]:
import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load models directly
print("Loading models...")
knn = joblib.load("knn/knn_product_classifier/knn_model.joblib")
model = SentenceTransformer("knn/knn_product_classifier/sentence_transformer")
print("Models loaded successfully!")

# Try to load training data info if available
try:
    train_info = pd.read_csv("knn/knn_product_classifier/train_data_info.csv")
    has_train_info = True
    print("Training data info loaded successfully!")
except:
    has_train_info = False
    print("Training data info not found. Only category information will be available.")

def predict_with_neighbors(query, n_neighbors=3):
    """
    Make a prediction and return category, confidence, and neighbor information
    
    Args:
        query (str): The product description to classify
        n_neighbors (int): Number of neighbors to return
    
    Returns:
        tuple: (category, confidence, neighbors)
    """
    # Preprocess the query
    processed_query = query.lower().strip()
    
    # Generate embedding
    query_embedding = model.encode([processed_query])
    
    # Get predictions and distances
    distances, indices = knn.kneighbors(query_embedding)
    distances = distances[0]
    indices = indices[0]
    
    # Get predicted category
    category = knn.predict(query_embedding)[0]
    
    # Calculate confidence
    max_distance = 2.0  # Maximum cosine distance
    confidence = 1 - (np.mean(distances) / max_distance)
    
    # Get neighbor information
    neighbors_info = []
    
    # Access neighbor information
    for idx, dist in zip(indices, distances):
        if has_train_info:
            # If we have the training data info
            title = train_info.loc[idx, 'title'] if 'title' in train_info.columns else f"Item {idx}"
            category_name = train_info.loc[idx, 'category_name'] if 'category_name' in train_info.columns else knn.classes_[knn._y[idx]]
        else:
            # Fallback to just using what we can get from KNN
            title = f"Item {idx}"
            # Get actual category name from KNN's classes_ attribute
            category_name = knn.classes_[knn._y[idx]] if hasattr(knn, '_y') else "Unknown"
        
        neighbors_info.append({
            'title': title,
            'category': category_name,
            'distance': dist,
            'similarity': 1 - (dist / max_distance),
            'index': idx  # Store the index for reference
        })
    
    return category, confidence, neighbors_info

# Make a prediction
query = "wireless bluetooth headphones"
category, confidence, neighbors = predict_with_neighbors(query)

# Print results
print(f"\nQuery: '{query}'")
print(f"Predicted Category: {category}")
print(f"Confidence: {confidence:.2f}")
        
print("\nSimilar Products:")
for i, neighbor in enumerate(neighbors, 1):
    print(f"{i}. {neighbor['title'][:80]}...")
    print(f"   Category: {neighbor['category']}")
    print(f"   Similarity: {neighbor['similarity']:.2f}")
    print()


Loading models...
Models loaded successfully!
Training data info not found. Only category information will be available.

Query: 'wireless bluetooth headphones'
Predicted Category: Other
Confidence: 0.76

Similar Products:
1. Item 1691...
   Category: Other
   Similarity: 0.83

2. Item 2417...
   Category: Other
   Similarity: 0.73

3. Item 4439...
   Category: Other
   Similarity: 0.72



In [ ]:
from flask import Flask, request, jsonify
import joblib
import os
from sentence_transformers import SentenceTransformer
import numpy as np

app = Flask(__name__)

# Load models
print("Loading models...")
knn = joblib.load("knn_product_classifier/knn_model.joblib")
model = SentenceTransformer("knn_product_classifier/sentence_transformer")
print("Models loaded successfully!")

def preprocess_text(text):
    """Basic text preprocessing"""
    if isinstance(text, str):
        return text.lower().strip()
    return ""

@app.route('/', methods=['GET'])
def home():
    return jsonify({
        "status": "online",
        "usage": {
            "endpoint": "/predict",
            "method": "POST",
            "body": {"query": "your product description here"}
        }
    })

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Get query from request
        data = request.json
        if not data or 'query' not in data:
            return jsonify({"error": "No query provided"}), 400
            
        query = data['query']
            
        # Process and predict
        processed_query = preprocess_text(query)
        query_embedding = model.encode([processed_query])
        
        # Get predictions and distances
        distances, indices = knn.kneighbors(query_embedding)
        distances = distances[0]
        
        # Get predicted category
        category = knn.predict(query_embedding)[0]
        
        # Calculate confidence
        max_distance = 2.0
        confidence = 1 - (np.mean(distances) / max_distance)
        
        return jsonify({
            "category": category,
            "confidence": float(confidence),
            "query": query
        })
    
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    port = int(os.environ.get('PORT', 5000))
    app.run(host='0.0.0.0', port=port)

c:\Users\Nikhil\Downloads\rag\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading models...
Models loaded successfully!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.101.177:5000
Press CTRL+C to quit
127.0.0.1 - - [31/Mar/2025 20:55:16] "POST /predict HTTP/1.1" 200 -


: 

In [2]:
import requests
import json

url = "https://product-classifier-api.onrender.com/predict"
data = {"query": "wireless earbuds with noise cancellation"}

response = requests.post(url, json=data)
result = response.json()

print(f"Category: {result['category']}")
print(f"Confidence: {result['confidence']:.2f}")

Category: Other
Confidence: 0.75
